# IA Generativa con Embeddings en MongoDB
Almacenamiento de preguntas frecuentes con búsqueda semántica basada en embeddings.

In [ ]:
# pip install pymongo sentence-transformers
from pymongo import MongoClient
from sentence_transformers import SentenceTransformer
import numpy as np

In [ ]:
# Conexión y configuración
client = MongoClient("mongodb://localhost:27017")
db = client["faq_db"]
col = db["questions"]

model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
# Insertar preguntas con embeddings
faq_questions = [
    "¿Cómo instalo Python?",
    "¿Qué es un modelo de lenguaje?",
    "Explica el aprendizaje profundo.",
    "¿Para qué sirve MongoDB?",
]

for q in faq_questions:
    emb = model.encode(q).tolist()
    col.insert_one({"question": q, "embedding": emb})

In [ ]:
# Consulta semántica
query = "¿Cómo se instala un lenguaje de programación?"
q_emb = model.encode(query)

def cos_sim(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

matches = []
for doc in col.find():
    sim = cos_sim(q_emb, doc['embedding'])
    matches.append((doc['question'], sim))

matches.sort(key=lambda x: x[1], reverse=True)
for q, s in matches[:3]:
    print(f"Similitud: {round(s, 2)} - '{q}'")